In [ ]:
import pathlib
import platform
import pickle

import haiku as hk
import jax
import numpy as np
import hvplot.xarray
import panel as pn
import xarray as xr
import cartopy.crs as ccrs
import grain.python as grain

from jax.sharding import PartitionSpec, NamedSharding

from graphcast import xarray_jax, xarray_tree, checkpoint, typed_graph
from graphcast.casting import Bfloat16Cast
from graphcast.data_utils import extract_inputs_targets_forcings
from graphcast.mesh_connectivity import get_connected_mesh_nodes, mask_mesh
from graphcast.mesh_graph import faces_to_edges, MeshGraph
from graphcast.model import TASK, TaskConfig, ModelConfig, GraphCast
from graphcast.normalization import InputsAndResiduals
from graphcast.mask import MaskedPredictor
from graphcast.ocean_mesh_utils import read_mesh
from graphcast.xarray_jax import unwrap_vars, unwrap_data
from graphcast.dataloader import ARCODataSource, AddLogDepthCoordinate

pn.extension()

jax.config.update("jax_traceback_filtering", 'off')

In [ ]:
if platform.node().endswith('leonardo.local'):
    root = pathlib.Path("/leonardo_scratch/large/userexternal/scampane/xcast/")
    resolution = "0p25"
    ocean_mesh_filename = "constant_coarse.msh"
    batch_size = 2
else:
    root = pathlib.Path("../")
    resolution = "1"
    ocean_mesh_filename = "constant_mini.msh"
    batch_size = 2
    jax.config.update('jax_num_cpu_devices', 4)

In [ ]:
def get_dashboard(dataset, title="", width=600, height=480, **kwargs):

    dataset = xarray_jax.to_np(dataset)
    dataset_min = dataset.min()
    dataset_max = dataset.max()

    variable_selector = pn.widgets.Select(
        name='Variable',
        options=list(dataset.data_vars))

    if 'level' in dataset.coords:
        level_options = {val: idx for idx, val in enumerate(dataset['level'].to_numpy())}
    else:
        level_options = []

    level_selector = pn.widgets.Select(
        name='Level',
        options=level_options)

    if 'time' in dataset.coords:
        time_options = {val: idx for idx, val in enumerate(dataset['time'].dt.days.to_numpy())}
    else:
        time_options = []
        
    time_selector = pn.widgets.DiscreteSlider(
        name='Day',
        options=time_options)
  
    batch_selector = pn.widgets.Select(
        name='Batch',
        options=[] if 'batch' not in dataset.dims else dataset['batch'].to_numpy().tolist())
    
    @pn.depends(variable_selector.param.value, level_selector.param.value, batch_selector.param.value, time_selector.param.value)
    def display(selected_variable, selected_level, selected_batch, selected_time):
        try:
            da = dataset[selected_variable]
            da_min = dataset_min[selected_variable]
            da_max = dataset_max[selected_variable]
            if 'level' in da.dims:
                da = da.isel(level=selected_level)
                da = da.drop_vars('level')
            if 'time' in da.dims:
                da = da.isel(time=selected_time)
                da = da.drop_vars('time')
            if 'batch' in da.dims:
                da = da.isel(batch=selected_batch)
            return da.hvplot.image("lon", "lat", clim=(da_min, da_max), width=width, height=height, **kwargs)
        except Exception as e:
            # Return an informative message if an error occurs during plotting
            return pn.pane.Markdown(f"### Error generating plot for var={selected_variable}, level={selected_level}, batch={selected_batch}, time={selected_time}: {e}")

    dashboard = pn.Row(
        pn.Column(
            title,
            variable_selector,
            level_selector,
            batch_selector,
            time_selector),
        display)
    
    return dashboard

In [ ]:
data_source = ARCODataSource(path=(root / f"data/dataset/dataset_tres-1d_res-{resolution}_levels-10_arco"), timesteps=3)
sampler = grain.IndexSampler(num_records=len(data_source), num_epochs=1, shuffle=True, seed=0)
operations = [AddLogDepthCoordinate(), 
              grain.Batch(batch_size=batch_size, drop_remainder=True, batch_fn=lambda datasets: xr.concat(datasets, dim='batch'))]
dataloader = grain.DataLoader(data_source=data_source, sampler=sampler, operations=operations, worker_count=0)

In [ ]:
for dataset in dataloader:
    break
dataset

In [ ]:
get_dashboard(dataset, title="# Data from ARCO-OCEAN", projection=ccrs.Robinson())

In [ ]:
devices = jax.local_devices()
devices

In [ ]:
devices_mesh = jax.make_mesh((2, 2), ('batch', 'graph'), devices=jax.local_devices())
devices_mesh

In [ ]:
def put_dataset(dataset, replicate_along_batch=False):
    if replicate_along_batch:
        sharding = NamedSharding(devices_mesh, PartitionSpec())
    else:
        sharding = NamedSharding(devices_mesh, PartitionSpec('batch'))

    def _put_dataarray(data_array):
        return jax.tree_util.tree_map(lambda xs: jax.device_put(xs, sharding), data_array)
        
    return dataset.map(lambda da: _put_dataarray(da))

In [ ]:
dataset_jax = xarray_jax.to_jax(dataset)
# Notice: as a side-effect, boolean variables get casted to float32 (which is useful)
dataset_jax = dataset_jax.fillna(value=jax.numpy.float32(0.0))

inputs, targets, forcings = extract_inputs_targets_forcings(dataset=dataset_jax, **TASK, target_lead_times="1d")

if devices_mesh is not None:
    inputs, targets, forcings = map(lambda x: put_dataset(x, replicate_along_batch=False), [inputs, targets, forcings])

In [ ]:
def visualize_sharding(xs: jax.Array):
    if xs.ndim > 2:
        n, *_ = xs.shape
        xs = xs.reshape(n, -1)
    jax.debug.visualize_array_sharding(xs)

In [ ]:
visualize_sharding(unwrap_data(inputs['zos']))

In [ ]:
inputs

In [ ]:
targets

In [ ]:
forcings

In [ ]:
ocean_mesh, boundary_nodes = read_mesh(root / f"data/geometry/{ocean_mesh_filename}")

ocean_mesh_graph = MeshGraph(vertices=ocean_mesh.vertices, edges=faces_to_edges(ocean_mesh.faces), faces=ocean_mesh.faces)

ocean_mesh_graph

In [ ]:
def get_max_edge_distance(mesh):
  senders, receivers = faces_to_edges(mesh.faces)
  edge_distances = np.linalg.norm(
      mesh.vertices[senders] - mesh.vertices[receivers], axis=-1)
  return edge_distances.max()

query_radius = 0.6 * get_max_edge_distance(ocean_mesh_graph)
query_radius

In [ ]:
connected_mesh_nodes = get_connected_mesh_nodes(grid_lat=data_source.mask['lat'],
                                                grid_lon=data_source.mask['lon'],
                                                mesh_graph=ocean_mesh_graph,
                                                grid_mask=data_source.mask,
                                                query_radius=query_radius,
                                                workers=-1)

In [ ]:
ocean_mesh_graph.vertices.shape[0], len(connected_mesh_nodes)

In [ ]:
connected_mesh_graph, _  = mask_mesh(connected_mesh_nodes, ocean_mesh_graph, mode='all')

In [ ]:
model_config = ModelConfig(
    latent_size=512,
    gnn_msg_steps=16,
    hidden_layers=1,
    radius_query_fraction_edge_length=0.6)

model_config

In [ ]:
with (root / 'data/model_config.ckpt').open('wb') as file:
    checkpoint.dump(file, model_config)

In [ ]:
task_config = TASK

task_config

In [ ]:
with (root / 'data/task_config.ckpt').open('wb') as file:
    checkpoint.dump(file, task_config)

The location and scale of the variables without `time` dimension (or computed analytically) have been computed as follows.

| Variable name      | Normalization |
|--------------------|---------------|
| `deptho`           | [0, 1]        |
| `waverys_deptho`   | [0, 1]        |
| `tisr`             | [0, 1]        |
| `z`                | [0, 1]        |
| `uparea`           | [0, 1]        |
| `glorys_mask`      | none          |
| `glofas_mask`      | none          |
| `lsm`              | none          |
| `year_progress_sin`| none          |
| `year_progress_cos`| none          |

In [ ]:
normalization_artifacts = xr.open_datatree(root / f"data/dataset/dataset_tres-1d_res-{resolution}_levels-10_normalization", engine='zarr')
normalization_artifacts

In [ ]:
mean_by_level = normalization_artifacts['/inputs/location'].dataset
mean_by_level = mean_by_level.fillna(0.0)

stddev_by_level = normalization_artifacts['/inputs/scale'].dataset
stddev_by_level = stddev_by_level.fillna(1.0)
stddev_by_level = stddev_by_level.clip(min=1e-18)

diffs_stddev_by_level = normalization_artifacts['/residuals/scale'].dataset
diffs_stddev_by_level = diffs_stddev_by_level.fillna(1.0)
diffs_stddev_by_level = diffs_stddev_by_level.clip(min=1e-18)

mean_by_level, stddev_by_level, diffs_stddev_by_level = map(lambda ds: put_dataset(ds, replicate_along_batch=True), [mean_by_level, stddev_by_level, diffs_stddev_by_level])

In [ ]:
visualize_sharding(unwrap_data(mean_by_level['zos']))

In [ ]:
# TODO: find out how tisr and progress variables are normalized in original GraphCast code

In [ ]:
get_dashboard(stddev_by_level, title='# Std by level', projection=ccrs.Robinson())

In [ ]:
# Deeper one-step predictor.
predictor = GraphCast(model_config, 
                      task_config, 
                      grid_lat=dataset['lat'].to_numpy(), 
                      grid_lon=dataset['lon'].to_numpy(), 
                      grid_mask=data_source.mask,
                      mesh_graph=connected_mesh_graph,
                      boundary_nodes=boundary_nodes,
                      ensure_divisible_by=4)

In [ ]:
def put_graph(graph):
    
    def _put_leaf(xs):
        pspec = jax.sharding.PartitionSpec() if len(xs.shape) == 1 else jax.sharding.PartitionSpec('graph')
        return jax.device_put(xs, jax.sharding.NamedSharding(devices_mesh, pspec))
        
    return jax.tree_util.tree_map(_put_leaf, graph)

for attr in ['_grid2mesh_graph_structure', '_mesh_graph_structure', '_mesh2grid_graph_structure']:
    setattr(predictor, attr, put_graph(getattr(predictor, attr))) 

In [ ]:
visualize_sharding(predictor._grid2mesh_graph_structure.nodes['grid_nodes'].features)

In [ ]:
with (root / 'data/model.pickle').resolve().open('wb') as file:
    pickle.dump(predictor, file, pickle.HIGHEST_PROTOCOL)

In [ ]:
# Modify inputs/outputs to `graphcast.GraphCast` to handle conversion to from/to float32 to/from BFloat16.
predictor = Bfloat16Cast(predictor)

# Modify inputs/outputs to `casting.Bfloat16Cast` so the casting to/from BFloat16 happens after applying normalization to the inputs/targets.
predictor = InputsAndResiduals(
    predictor,
    diffs_stddev_by_level=diffs_stddev_by_level,
    mean_by_level=mean_by_level,
    stddev_by_level=stddev_by_level)

# Mask inputs/outputs replacing missing values with 0.0
predictor = MaskedPredictor(predictor, mask=dataset['glorys_mask'], value=0.0)

In [ ]:
@hk.without_apply_rng
@hk.transform
def run_forward(inputs, targets_template, forcings):
  return predictor(inputs, targets_template=targets_template, forcings=forcings)

run_forward_jit = jax.jit(jax.checkpoint(run_forward.apply, policy=jax.checkpoint_policies.nothing_saveable))

In [ ]:
key = jax.random.key(0)

params = run_forward.init(rng=key, inputs=inputs, targets_template=targets, forcings=forcings)

In [ ]:
params = jax.tree_util.tree_map(lambda xs: jax.device_put(xs, NamedSharding(devices_meshes, PartitionSpec())), params)

In [ ]:
forecast = run_forward_jit(params=params, inputs=inputs, targets_template=targets, forcings=forcings)

In [ ]:
get_dashboard(forecast, title="# Forecast", projection=ccrs.Robinson())

In [ ]:
@hk.without_apply_rng
@hk.transform
def loss_fn(inputs, targets, forcings):
  loss, diagnostics = predictor.loss(inputs, targets, forcings=forcings, levels_normalization_coord='log-depth')
  return xarray_tree.map_structure(
      lambda x: unwrap_data(x.mean(), require_jax=True),
      (loss, diagnostics))

loss_fn_apply_jit = jax.jit(loss_fn.apply)

In [ ]:
loss, diagnostics = loss_fn_apply_jit(params, inputs, targets, forcings)
loss